## In this notebook:
#### We import PET-derived volumetric receptor maps and parcellate using the Schaefer100 atlas.

1. Import packages and maps
2. Parcellate


### Import Packages

In [1]:
import pandas as pd
from nilearn.datasets import fetch_atlas_schaefer_2018
from neuromaps.parcellate import Parcellater
import warnings
from neuromaps.datasets import fetch_fslr, fetch_annotation
from neuromaps import datasets, images, nulls, resampling, stats, transforms, parcellate
from neuromaps.parcellate import Parcellater 
from neuromaps.resampling import resample_images
from neuromaps.images import dlabel_to_gifti
import neuromaps
import nibabel as nib
import os
import numpy as np
import mayavi
from PIL import Image
import hcp_utils as hcp
import nilearn.plotting as plotting
import warnings
from netneurotools.datasets import fetch_cammoun2012
from netneurotools import datasets as nntdata
#import abagen
#import abagen ## This is important for certification purposes 
warnings.filterwarnings('ignore')

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


### Set Variables

In [2]:
schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
#cammoun = fetch_cammoun2012()
#desikan = '/Users/pecsok/nilearn_data/desikan_killiany/desikan-killiany2006_rois_1mm.nii.gz'
#desikan = '/Users/pecsok/nilearn_data/desikan_killiany/desikanKillianyMNI.nii.gz'
atlas = 'schaefer'

#scale = 'scale500'
scale = 'scale1000_17'
#path = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/"
path = "/Users/pecsok/neuromaps-data/annotations/sydnor2021/SAaxis/fsLR"
outpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/"
SAleft = os.path.join(path,"source-sydnor2021_desc-SAaxis_space-fsLR_den-32k_hemi-L_feature.func.gii")
SAright = os.path.join(path,"source-sydnor2021_desc-SAaxis_space-fsLR_den-32k_hemi-R_feature.func.gii")

schaefer_dscalar = '/Users/pecsok/neuromaps-data/atlases/fsLR/Schaefer2018_1000Parcels_17Networks_order.dscalar.nii'
Lschaefer_gii = '/Users/pecsok/Desktop/schaefergiftiL.gii' 
#print(schaefer['maps'])
print(SAleft)
schaefer_newgii = '/Users/pecsok/neuromaps-data/atlases/fsLR/Schaefer2018_1000parcels_17N.gii'



/Users/pecsok/neuromaps-data/annotations/sydnor2021/SAaxis/fsLR/source-sydnor2021_desc-SAaxis_space-fsLR_den-32k_hemi-L_feature.func.gii


In [3]:
from neuromaps import images, transforms, datasets
from neuromaps.parcellate import Parcellater
import netneurotools.datasets as nntdata

saaxis = datasets.fetch_annotation(source='sydnor2021', desc='SAaxis', space='fsLR', den='32k')
# transform saaxis from fslr to fsaverage
saaxis_surface10k = transforms.fslr_to_fsaverage(saaxis, target_density='10k',
                                                  hemi=None,
                                                  method='linear')

schaefer = nntdata.fetch_schaefer2018(version='fsaverage5')['1000Parcels7Networks']
parc_gifti = images.annot_to_gifti(schaefer)
parc = Parcellater(parc_gifti, 'fsaverage')
parcellated_saaxis = parc.fit_transform(saaxis_surface10k, 'fsaverage')

# parcellated values aren't integers, but we can get their rankings instead
order = parcellated_saaxis.argsort()
parcellated_sa_ranks = order.argsort()

In [4]:
parcellated_sa_ranks.shape

(1000,)

In [56]:
# Convert schaefer from dlabel to gifti
# Import dlabel file by hand 
schaefer_dlabel = '/Users/pecsok/neuromaps-data/atlases/fsLR/Schaefer2018_1000Parcels_17Networks_order.dlabel.nii'
saaxis = fetch_annotation(source='sydnor2021', desc='SAaxis', space='fsLR', den='32k')
#schaefer = nntdata.fetch_schaefer2018('fslr32k')['1000Parcels7Networks']
print(schaefer)
parc = Parcellater(dlabel_to_gifti(schaefer_dlabel), 'fsLR')
saaxis_parc = parc.fit_transform(saaxis, 'fsLR')


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


/Users/pecsok/nnt-data/atl-schaefer2018/fslr32k/atl-Schaefer2018_space-fslr32k_hemi-LR_desc-1000Parcels7Networks_deterministic.dlabel.nii


IndexError: index 1000 is out of bounds for axis 0 with size 1000

In [58]:
# Convert schaefer from dlabel to gifti
saaxis = fetch_annotation(source='sydnor2021', desc='SAaxis', space='fsLR', den='32k')
schaefer = nntdata.fetch_schaefer2018('fslr32k')['1000Parcels7Networks']
parc = Parcellater(dlabel_to_gifti(schaefer), 'fsLR')
saaxis_parc = parc.fit_transform(saaxis, 'fsLR')


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


IndexError: index 1000 is out of bounds for axis 0 with size 1000

### Load in maps of interest

In [3]:
annotations_nii = [SAleft] #schaefer_surf, 

receptors_nii = [path+'5HT1a_way_hc36_savli.nii',
                 path+'5HT1a_cumi_hc8_beliveau.nii',
                 path+'VAChT_feobv_hc18_aghourian_sum.nii']

In [16]:
def split_gii_into_hemispheres(input_gii, left_output_gii, right_output_gii):
    # Load the GIFTI file
    gii = nib.load(input_gii)
    data = gii.darrays[0].data  # Assuming data is in the first darray

    # Split the data (assumes data is split by vertices for left and right hemisphere)
    mid_point = data.shape[0] // 2
    left_data = data[:mid_point]
    right_data = data[mid_point:]

    # Create new GIFTI structures for left and right hemisphere data
    left_gii = nib.GiftiImage(darrays=[nib.gifti.GiftiDataArray(left_data)])
    right_gii = nib.GiftiImage(darrays=[nib.gifti.GiftiDataArray(right_data)])

    # Save the split files
    nib.save(left_gii, left_output_gii)
    nib.save(right_gii, right_output_gii)
    return left_output_gii, right_output_gii

# Example usage
left_gii, right_gii = split_gii_into_hemispheres(
    input_gii=schaefer_newgii,
    left_output_gii='/Users/pecsok/neuromaps-data/atlases/fsLR/left_schaefer1000.gii',
    right_output_gii='/Users/pecsok/neuromaps-data/atlases/fsLR/right_schaefer1000.gii'
)
print(left_gii)

/Users/pecsok/neuromaps-data/atlases/fsLR/left_schaefer1000.gii


In [37]:
gii = nib.load(left_gii)
def inspect_gii(gii_file):
    gii = nib.load(gii_file)
for i, darray in enumerate(gii.darrays):
    print(f"DataArray {i}:")
    print(f"  Shape: {darray.data.shape}")
    print(f"  Data type: {darray.data.dtype}")

#print(inspect_gii(schaefer_surf))
#print(inspect_gii(schaefer_newgii))
#print(inspect_gii(Lschaefer_gii))
print(inspect_gii(SAleft))

DataArray 0:
  Shape: (32492,)
  Data type: float32
None


### Parcellate

In [18]:
if atlas == 'schaefer': 
    parcellated = {}
    parcellater = Parcellater(left_gii, 'fsLR', hemi='L')
    print("yes")
    #parcellater = Parcellater(schaefer_surf, 'fsLR', hemi='L')
    for annotation in annotations_nii:
        parcellated[annotation] = parcellater.fit_transform(annotation, 'fsLR', True, hemi='L')
        #print(parcellated[receptor])
        print(parcellated[annotation].shape)
        #name = receptor.split('/')[-1]  # get nifti file name
        #name = name.split('.')[0]  # remove .nii
        #np.savetxt(outpath+name+'.csv', parcellated[receptor], delimiter=',')

yes


IndexError: boolean index did not match indexed array along dimension 0; dimension is 32492 but corresponding boolean dimension is 64984

In [5]:
#print(parcellated['/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/GABAa-bz_flumazenil_hc16_norgaard.nii'])

"""

elif atlas == 'cammoun':
    parcellated = {}
    parcellater = Parcellater(cammoun[scale], 'MNI152')
    for receptor in receptors_nii:
        parcellated[receptor] = parcellater.fit_transform(receptor, 'MNI152', True)
        print(parcellated[receptor])
        print(parcellated[receptor].shape)
        name = receptor.split('/')[-1]  # get nifti file name
        name = name.split('.')[0]  # remove .nii
        np.savetxt(outpath+name+'.csv', parcellated[receptor], delimiter=',')

elif atlas == 'desikan':
    parcellated = {}
    parcellater = Parcellater(desikan, 'MNI152')
    for receptor in receptors_nii:
        parcellated[receptor] = parcellater.fit_transform(receptor, 'MNI152', True)
        #print(parcellated[receptor])
        print(parcellated[receptor].shape)
        name = receptor.split('/')[-1]  # get nifti file name
        name = name.split('.')[0]  # remove .nii
        np.savetxt(outpath+name+'.csv', parcellated[receptor], delimiter=',')

"""
#print(parcellated)

KeyError: '/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/GABAa-bz_flumazenil_hc16_norgaard.nii'

In [ ]:
# Desikan Killiany Atlas
